# Camera-depth training and validation analysis

Ce notebook lance optionnellement un entraînement KITTI, charge les métriques par époque, compare le modèle à la profondeur splattée et inspecte une prédiction du split de validation. La logique réutilisable reste dans `rgb_lidar_fusion.camera_depth_analysis`.

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from rgb_lidar_fusion.camera_depth_analysis import (
    analyze_camera_depth_sample,
    compare_camera_depth_holdout_checkpoints,
    has_camera_depth_validation_metrics,
    load_camera_depth_checkpoint,
    load_camera_depth_metrics,
    plot_camera_depth_ablation,
    plot_camera_depth_holdout_input,
    plot_camera_depth_holdout_comparison,
    plot_camera_depth_metrics,
    plot_camera_depth_sample,
    require_gpu_ready,
    run_gpu_preflight,
)
from rgb_lidar_fusion.dataset import KittiObjectDepthDataset
from rgb_lidar_fusion.lidar_splatting import SplattingConfig
from rgb_lidar_fusion.training import (
    camera_depth_holdout_options,
    load_training_config,
    prepare_camera_depth_evaluation_batch,
    run_kitti_camera_depth_training,
    split_dataset_indices,
)

## Paramètres du run

L'entraînement est désactivé par défaut. Passer `RUN_TRAINING` à `True` pour créer le run d'analyse. `EPOCHS` est le nombre total d'époques visé, y compris lors d'une reprise. Le batch notebook reste à 1 par défaut pour conserver une marge sur le GPU local de 4 Gio. Avec `REQUIRE_CUDA=True`, un préflight valide est obligatoire avant l'entraînement.

In [ ]:
CONFIG_PATH = ROOT / "configs/training/kitti_camera_depth_random_holdout_512.yaml"
RUN_OUTPUT_DIR = ROOT / "results/training/kitti_camera_depth_random_holdout_512"
SPARSE_ABLATION_OUTPUT_DIR = ROOT / "results/training/kitti_camera_depth_sparse_ablation"
RUN_TRAINING = False
RUN_SPLATTING_ABLATION = False
RUN_HOLDOUT_COMPARISON = False
HOLDOUT_COMPARISON_CONFIG_PATH = ROOT / "configs/training/kitti_camera_depth_block_dropout.yaml"
HOLDOUT_CHECKPOINTS = {
    "random_points": ROOT / "results/training/kitti_camera_depth_random_holdout_512/checkpoints/latest.pt",
    "spatial_blocks": ROOT / "results/training/kitti_camera_depth_block_dropout_512/checkpoints/latest.pt",
}
HOLDOUT_COMPARISON_OUTPUT = ROOT / "results/analysis/camera_depth_holdout_comparison.json"
PREVIEW_SAMPLE_INDEX = 0
RESUME_EXISTING = False
EPOCHS = 5
BATCH_SIZE = 1
DEVICE = "cuda"
REQUIRE_CUDA = True
MINIMUM_FREE_GPU_MEMORY_MIB = 2500
VALIDATION_SAMPLE_POSITION = 0

config = load_training_config(CONFIG_PATH)
data_root = Path(str(config["data_root"]))
if not data_root.is_absolute():
    config["data_root"] = str((ROOT / data_root).resolve())
config.update(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    lidar_mode="enriched",
    output_dir=str(RUN_OUTPUT_DIR),
)
checkpoint_path = RUN_OUTPUT_DIR / "checkpoints/latest.pt"
metrics_path = RUN_OUTPUT_DIR / "metrics.json"
config

## Aperçu du holdout, sans entraînement

Ces vues CPU montrent les régions sélectionnées, les mesures conservées et retirées, puis le splatting recalculé. Elles fonctionnent avant tout checkpoint. Le RGB reste visible ; seuls les points LiDAR dans les régions sont retirés. Les paramètres de blocs proviennent de la configuration sélectionnée, avec les valeurs par défaut si elle ne les précise pas.

In [ ]:
if Path(config["data_root"]).is_dir():
    preview_dataset = KittiObjectDepthDataset(
        config["data_root"],
        image_shape=(int(config["height"]), int(config["width"])),
        max_depth_m=float(config["max_depth_m"]),
    )
    for mode in ("random_points", "spatial_blocks"):
        preview_batch = prepare_camera_depth_evaluation_batch(
            dataset=preview_dataset, indices=[PREVIEW_SAMPLE_INDEX],
            holdout_fraction=float(config["holdout_fraction"]),
            holdout_seed=int(config.get("validation_holdout_seed", config.get("split_seed", config.get("seed", 0)))),
            splatting_config=SplattingConfig(
                radius_px=int(config["splat_radius_px"]),
                sigma_px=float(config["splat_sigma_px"]),
            ),
            **camera_depth_holdout_options({**config, "holdout_mode": mode}),
        )
        removed = int(preview_batch["loss_mask"].sum())
        valid = int((preview_batch["depth_target"] > 0).sum())
        print(f"{mode}: removed={removed}/{valid} ({removed / valid:.1%})")
        preview_figure = plot_camera_depth_holdout_input(
            preview_batch, max_depth_m=float(config["max_depth_m"]),
        )
        preview_figure.suptitle(mode)
        display(preview_figure)
        plt.close(preview_figure)
else:
    print(f"No KITTI data at {config['data_root']}")

## Préflight GPU

Ce contrôle vérifie l'état de récupération NVIDIA, la mémoire libre, les autres processus CUDA et exécute un petit calcul CUDA synchronisé. Entre deux runs terminés, il libère d'abord le cache PyTorch inutilisé et affiche la mémoire des tenseurs actifs séparément de la mémoire réservée. Il ne supprime pas les tenseurs encore référencés et ne doit pas être exécuté pendant un entraînement. Si le rapport échoue, ne pas lancer l'entraînement ; une action `Reset` après une erreur Xid nécessite de redémarrer la machine.

In [ ]:
gpu_report = None
if DEVICE != "cpu":
    gpu_report = run_gpu_preflight(
        minimum_free_memory_mib=MINIMUM_FREE_GPU_MEMORY_MIB,
        run_smoke_test=True,
        release_cached_memory=True,
    )
    print(gpu_report.format())
else:
    print("GPU preflight skipped because DEVICE='cpu'.")

In [ ]:
if RUN_TRAINING and REQUIRE_CUDA and DEVICE != "cpu":
    if gpu_report is None:
        raise RuntimeError("Run the GPU preflight cell before training.")
    require_gpu_ready(gpu_report)

if RUN_TRAINING:
    run_config = dict(config)
    if RESUME_EXISTING:
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"Cannot resume missing checkpoint: {checkpoint_path}")
        run_config["resume_from"] = str(checkpoint_path)
    result = run_kitti_camera_depth_training(run_config)
    print(result.device_diagnostic)
    print(f"epochs_completed={result.epochs_completed}")
    print(f"checkpoint={result.checkpoint_path}")
    if result.metrics:
        display(result.metrics[-1])
else:
    print("Training skipped. Set RUN_TRAINING=True to create or extend the analysis run.")

## Courbes et comparaison au splatting

La loss est exprimée en profondeur normalisée. Les MAE/RMSE sont exprimées en mètres sur les mêmes pixels LiDAR holdout pour le réseau et la baseline `depth_expanded`.

In [ ]:
metrics = []
if metrics_path.is_file():
    metrics = load_camera_depth_metrics(metrics_path)
    if has_camera_depth_validation_metrics(metrics):
        metric_figure = plot_camera_depth_metrics(metrics)
        display(metric_figure)
        plt.close(metric_figure)
    else:
        print(
            "This run predates validation metrics. Create a new run with "
            "RUN_TRAINING=True before plotting train/validation curves."
        )
else:
    print(f"No metrics yet at {metrics_path}")

In [ ]:
if has_camera_depth_validation_metrics(metrics):
    latest = metrics[-1]
    mae_gain_m = latest["val_splat_mae_m"] - latest["val_mae_m"]
    rmse_gain_m = latest["val_splat_rmse_m"] - latest["val_rmse_m"]
    print(f"epoch={latest['epoch']}")
    print(f"model MAE={latest['val_mae_m']:.3f} m | splat MAE={latest['val_splat_mae_m']:.3f} m")
    print(f"model RMSE={latest['val_rmse_m']:.3f} m | splat RMSE={latest['val_splat_rmse_m']:.3f} m")
    print(f"gain over splat: MAE={mae_gain_m:+.3f} m, RMSE={rmse_gain_m:+.3f} m")
    print(f"splat coverage={latest['val_splat_coverage']:.1%} | validation pixels={latest['val_pixel_count']}")
    if latest.get("val_holdout_fraction") is not None:
        print(f"achieved validation holdout={latest['val_holdout_fraction']:.1%}")

## Ablation : apport des canaux splattés

Cette expérience compare le run principal `enriched` (6 cartes sparse + `depth_expanded` + `confidence`) à un nouveau run `sparse` limité aux 6 cartes initiales. Les seeds, le split, les masques holdout et les hyperparamètres restent identiques ; seule l'entrée LiDAR du réseau change. Passer `RUN_SPLATTING_ABLATION=True` pour entraîner le run sparse depuis zéro.

In [ ]:
sparse_metrics_path = SPARSE_ABLATION_OUTPUT_DIR / "metrics.json"
if RUN_SPLATTING_ABLATION and REQUIRE_CUDA and DEVICE != "cpu":
    if gpu_report is None:
        raise RuntimeError("Run the GPU preflight cell before the ablation.")
    require_gpu_ready(gpu_report)

if RUN_SPLATTING_ABLATION:
    sparse_config = {
        **config,
        "lidar_mode": "sparse",
        "output_dir": str(SPARSE_ABLATION_OUTPUT_DIR),
    }
    sparse_result = run_kitti_camera_depth_training(sparse_config)
    print(sparse_result.device_diagnostic)
    print(f"sparse epochs_completed={sparse_result.epochs_completed}")
    print(f"sparse checkpoint={sparse_result.checkpoint_path}")
else:
    print("Sparse ablation skipped. Set RUN_SPLATTING_ABLATION=True to run it.")

In [ ]:
ablation_metrics = {}
if has_camera_depth_validation_metrics(metrics):
    ablation_metrics["enriched"] = metrics
if sparse_metrics_path.is_file():
    sparse_metrics = load_camera_depth_metrics(sparse_metrics_path)
    if has_camera_depth_validation_metrics(sparse_metrics):
        ablation_metrics["sparse"] = sparse_metrics

if set(ablation_metrics) == {"sparse", "enriched"}:
    best_by_mode = {
        mode: min(rows, key=lambda row: row["val_mae_m"])
        for mode, rows in ablation_metrics.items()
    }
    for mode in ("sparse", "enriched"):
        best = best_by_mode[mode]
        print(
            f"{mode:8s} best_epoch={best['epoch']} "
            f"MAE={best['val_mae_m']:.3f} m RMSE={best['val_rmse_m']:.3f} m"
        )
    mae_gain_m = (
        best_by_mode["sparse"]["val_mae_m"]
        - best_by_mode["enriched"]["val_mae_m"]
    )
    print(f"splatting MAE gain={mae_gain_m:+.3f} m (positive means enriched is better)")
    ablation_figure = plot_camera_depth_ablation(ablation_metrics)
    display(ablation_figure)
    plt.close(ablation_figure)
else:
    missing = {"sparse", "enriched"} - set(ablation_metrics)
    print(f"Missing ablation metrics for: {', '.join(sorted(missing))}")

## Inspection d'une frame de validation

Cette vue reconstruit le split et le holdout déterministe à partir de la configuration sauvegardée dans le checkpoint. En mode spatial, les régions retirées apparaissent en rouge sur le RGB. L'erreur affichée n'existe qu'aux pixels LiDAR retirés des entrées.

In [ ]:
if checkpoint_path.is_file():
    loaded = load_camera_depth_checkpoint(checkpoint_path, device=DEVICE)
    sample_config = {**config, **loaded.config}
    sample_config["holdout_mode"] = loaded.config.get("holdout_mode", "random_points")
    dataset = KittiObjectDepthDataset(
        ROOT / str(sample_config["data_root"]),
        image_shape=(int(sample_config["height"]), int(sample_config["width"])),
        max_depth_m=float(sample_config["max_depth_m"]),
        sample_limit=int(sample_config["sample_count"]),
    )
    _, validation_indices = split_dataset_indices(
        len(dataset),
        validation_fraction=float(sample_config["validation_fraction"]),
        seed=int(sample_config.get("split_seed", sample_config.get("seed", 0))),
    )
    dataset_index = validation_indices[VALIDATION_SAMPLE_POSITION]
    analysis = analyze_camera_depth_sample(
        model=loaded.model,
        dataset=dataset,
        dataset_index=dataset_index,
        holdout_fraction=float(sample_config["holdout_fraction"]),
        holdout_seed=int(sample_config.get("validation_holdout_seed", sample_config.get("split_seed", sample_config.get("seed", 0)))),
        splatting_config=SplattingConfig(
            radius_px=int(sample_config["splat_radius_px"]),
            sigma_px=float(sample_config["splat_sigma_px"]),
        ),
        device=loaded.device,
        max_depth_m=float(sample_config["max_depth_m"]),
        **camera_depth_holdout_options(sample_config),
    )
    print(f"checkpoint epoch={loaded.epoch}, step={loaded.step}, sample={analysis.sample_id}, holdout={analysis.holdout_mode}")
    sample_figure = plot_camera_depth_sample(analysis)
    display(sample_figure)
    plt.close(sample_figure)
else:
    print(f"No checkpoint yet at {checkpoint_path}")

## Validation croisée : random points et blocs spatiaux

`RUN_HOLDOUT_COMPARISON=True` évalue deux checkpoints déjà entraînés sur les deux modes de holdout ; aucun entraînement n'est lancé ici. Les configs `kitti_camera_depth_random_holdout_512.yaml` et `kitti_camera_depth_block_dropout.yaml` préparent les deux runs comparables. La comparaison vérifie dataset, split, seeds, normalisation, paramètres d'apprentissage et budget d'entraînement. Les checkpoints doivent avoir la même époque et le même nombre de steps. Les matrices comparent les checkpoints fournis, pas les meilleurs scores historiques.

Dans chaque colonne, les deux réseaux voient exactement les mêmes points conservés et cibles. Les mesures random et spatial restent distinctes. Une faible couverture splattée signifie que la baseline dispose de peu de profondeur propagée aux pixels retirés ; les erreurs de la baseline utilisent alors aussi les zéros des pixels non couverts.

In [ ]:
if RUN_HOLDOUT_COMPARISON:
    if REQUIRE_CUDA and DEVICE != "cpu":
        if gpu_report is None:
            raise RuntimeError("Run the GPU preflight before evaluation.")
        require_gpu_ready(gpu_report)
    comparison_config = load_training_config(HOLDOUT_COMPARISON_CONFIG_PATH)
    comparison_config["data_root"] = str((ROOT / str(comparison_config["data_root"])).resolve())
    comparison_config["batch_size"] = BATCH_SIZE
    comparison_rows = compare_camera_depth_holdout_checkpoints(
        HOLDOUT_CHECKPOINTS, evaluation_config=comparison_config, device=DEVICE, project_root=ROOT,
    )
    for row in comparison_rows:
        print(
            f"train={row['training_holdout_mode']:14s} eval={row['evaluation_holdout_mode']:14s} "
            f"MAE={row['mae_m']:.3f} m RMSE={row['rmse_m']:.3f} m "
            f"holdout={row['holdout_fraction']:.1%} splat coverage={row['splat_coverage']:.1%}"
        )
    comparison_figure = plot_camera_depth_holdout_comparison(comparison_rows)
    display(comparison_figure)
    plt.close(comparison_figure)
    HOLDOUT_COMPARISON_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    HOLDOUT_COMPARISON_OUTPUT.write_text(json.dumps({
        "evaluation_config": comparison_config, "rows": comparison_rows,
    }, indent=2) + "\n")
    print(f"Saved comparison to {HOLDOUT_COMPARISON_OUTPUT}")
else:
    print("Holdout comparison skipped. Set RUN_HOLDOUT_COMPARISON=True after both checkpoints exist.")

## Lecture des résultats

- `loss` et `val_loss` qui diminuent ensemble indiquent un apprentissage cohérent sur les points supervisés.
- Une loss train qui baisse tandis que la validation stagne ou remonte indique un surapprentissage.
- Un gain MAE/RMSE positif signifie que le réseau bat la profondeur splattée sur les pixels holdout.
- Ces mesures restent sparse : elles ne prouvent pas la qualité de la profondeur aux pixels dépourvus de vérité terrain LiDAR.